# 03 · Bosonic 表示新手教程

**Bosonic（高斯叠加）表示**把态写成若干高斯组件：

$$
\{(V_k, \bar r_k, w_k)\}_{k=1}^K
$$

权重 $w_k$ 可复；纯态归一要 **Gram**（组件不正交时 $\sum w$ 有讲究）。

本教程：`even_cat`、`gkp0`/`gkp1`、门、loss、logical overlap。

配套笔记：`03-Bosonic表示原理.md`。

## 1. 这是啥 / 为啥用

- Cat：$|\alpha\rangle \pm |-\alpha\rangle$ → 对角高斯 + **交叉项**（复均值组件）
- GKP：格子上许多窄峰 + 可选交叉 → 教学纠错码态
- **高斯门**：每个组件用同一 $S,d$ 变；**权重 $w$ 不变**
- 比 Fock 截断更适合「中等非高斯 + 仍近似高斯峰」

**一句话：** 非高斯，但还能拆成有限个高斯包时，用 Bosonic。

## 2. 约定

- 与 G 相同：$\hbar=1$，xxpp，真空 $V=I/2$
- $\sum_k w_k = 1$（可检）
- GKP：$\Delta=\sqrt{2\pi}$；`lattice=1d|2d`；`cross=none|nn|full`（2d 无 nn）
- 权重形式 $Z=c^\dagger S c$（Gram）——与 cat 同构思想

In [ ]:
# 从仓库根启动 Jupyter 最稳；若在 tutorials/ 里打开，这里兜底加路径
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "cvsim").is_dir():
    ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['SimHei']  # 中文支持
matplotlib.rcParams['axes.unicode_minus'] = False    # 负号显示
print("repo root:", ROOT)
print("numpy", np.__version__)

In [ ]:
from cvsim.bosonic import (
    even_cat,
    gkp0,
    gkp1,
    gkp_logical_overlap,
    homodyne_condition,
    loss,
    mean_photon,
    phase,
    squeeze,
    weight_sum,
)

## 3. 最小闭环：even cat

$|\mathrm{cat}_+\rangle \propto |\alpha\rangle + |-\alpha\rangle$ → **4 组件**（2 对角 + 2 交叉）。

In [ ]:
alpha = 0.8
cat = even_cat(alpha)
print("K =", cat.n_components)
print("sum w =", weight_sum(cat))
for i, c in enumerate(cat.components):
    print(f"  [{i}] w={c.w.real:+.5f}{c.w.imag:+.5f}j  rbar={c.rbar}")

## 4. 数字检查：门保持权重；phase 转峰

高斯门不改 $w$（仅改各组件的 $V,\bar r$）。

In [ ]:
cat = even_cat(0.8)
w0 = [c.w for c in cat.components]
cat2 = squeeze(phase(cat, 0.3), 0.1)
w1 = [c.w for c in cat2.components]
print("weights unchanged?", all(abs(a - b) < 1e-14 for a, b in zip(w0, w1)))
print("sum w after gates:", weight_sum(cat2))

## 5a. GKP 教学态

- `cross="none"`：只有对角峰（混态齿梳感）
- `cross="full"`：全齿对交叉（1d 更「纯态感」）
- `lattice="2d"`：$(x,p)$ 方格对角峰；`cross="full"` 可开但组件数 $M^2$

`gkp_logical_overlap`：用对角峰近似逻辑重叠；self ≈ 1，0 vs 1 应较小。

In [ ]:
eps, N = 0.12, 2
z0_none = gkp0(eps, grid_size=N, cross="none")
z0_full = gkp0(eps, grid_size=N, cross="full")
z1_full = gkp1(eps, grid_size=N, cross="full")
print("K none/full:", z0_none.n_components, z0_full.n_components)
print("sum w full:", weight_sum(z0_full))
print("<0|0> ~", gkp_logical_overlap(z0_full, z0_full))
print("<0|1> ~", gkp_logical_overlap(z0_full, z1_full), "  (|ov| should be smaller)")

z2 = gkp0(0.15, grid_size=1, lattice="2d", cross="none")
print("2d diag K:", z2.n_components, "expect 9")

## 5b. loss 与 condition 一瞥

`loss(T=0)` 理想全丢 → 近似真空，$\langle n\rangle\approx 0$，∑w 仍 1。

`homodyne_condition(state, mode, phi, outcome)` 在 B 上走 **复仿射 + 似然**（比 G 更一般）。

In [ ]:
cat = even_cat(0.8)
print("<n> cat:", mean_photon(cat))
print("<n> after T=0 loss:", mean_photon(loss(cat, T=0.0)))
post = homodyne_condition(cat, mode=0, phi=0.0, outcome=0.0)
print("K after condition:", post.n_components, "sum w:", weight_sum(post))

## 6. 诚实边界 + 何时换表示

**Bosonic 适合**

- Cat / 截断 GKP / 有限高斯叠加
- 与 G 共用辛门，但可保留交叉干涉

**诚实：本包 GKP 不是完整纠错栈**

- 无逻辑 Clifford 完备、无 dual 基展开、无 stabilizer 解码
- 2d `cross=nn` 未做；大 N 组件爆炸

**选型速查**

| 问题 | 优先 |
|------|------|
| 大规模线性光学、矩、loss | Gaussian |
| 光子数、Kerr、小系统 Wigner | Fock |
| Cat/GKP、有限非高斯叠加 | Bosonic |

跨表示同一数字：命令行 `python -m cvsim.demos.m4_cross_rep`。

## 自检

In [ ]:
cat = even_cat(0.8)
assert cat.n_components == 4
assert abs(weight_sum(cat) - 1.0) < 1e-12
z0 = gkp0(0.15, 2, cross="full")
assert z0.n_components == 25
assert abs(weight_sum(z0) - 1.0) < 1e-12
assert abs(gkp_logical_overlap(z0, z0) - 1.0) < 1e-8
z1 = gkp1(0.15, 2, cross="full")
assert abs(gkp_logical_overlap(z0, z1)) < 0.5
assert abs(mean_photon(loss(cat, 0.0))) < 1e-10
print("T3 self-check OK")